In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/janadarwish/sprints-dataset/employer_partner_requirements.json
/kaggle/input/datasets/janadarwish/sprints-dataset/sprints_cohort_data_2023.csv
/kaggle/input/datasets/janadarwish/cleaned/sprints_cohort_cleaned.csv


# 02 — Feature Engineering with NumPy

## Objective
Transform the cleaned learner dataset into useful numerical features using NumPy vectorization.

## Tasks
- Load the cleaned dataset from Notebook 01.
- Explore module scores and their missing values.
- Calculate a `Skill_Mastery_Index` for each learner using available module scores.
- Use NumPy vectorized operations to create additional analytical features.
- Explore relationships between learner performance and engagement.
- Save the engineered dataset for further analysis and visualization.

## Tools
- Pandas
- NumPy

In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv("/kaggle/input/datasets/janadarwish/cleaned/sprints_cohort_cleaned.csv")

In [4]:
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (24, 18)


,learner_id,cohort_id,enrollment_date,graduation_date,demographic_group,track_name,engagement_score,module_1_score,module_2_score,module_3_score,module_4_score,module_5_score,module_6_score,module_7_score,module_8_score,module_9_score,module_10_score,capstone_score
0,LRN-1001,2023-Q1-DA,2023-01-15,2023-07-20,Group_A,Advanced Data Analysis,85.0,90,88.0,92.0,85,89.0,94.0,88.0,91.0,95.0,90.0,93.0
1,LRN-1002,2023-Q1-DA,2023-01-15,2023-07-20,Group_B,Advanced Data Analysis,NaN,70,75.0,NaN,65,80.0,77.0,81.0,72.0,68.0,74.0,76.0
2,LRN-1003,2023-Q2-AI,2023-04-10,2023-10-15,Group_A,AI & Machine Learning,92.0,95,96.0,91.0,98,94.0,99.0,90.0,92.0,95.0,97.0,96.0
3,LRN-1004,2023-Q1-FS,2023-01-15,2023-07-20,Group_C,Full Stack Web Development,45.0,60,55.0,58.0,62,59.0,61.0,65.0,54.0,55.0,60.0,58.0
4,LRN-1005,2023-Q2-DA,2023-04-10,2023-10-15,Group_D,Advanced Data Analysis,78.0,80,82.0,79.0,85,81.0,NaN,86.0,88.0,82.0,84.0,85.0


In [5]:
score_columns = [
    f"module_{i}_score" for i in range(1, 11)
]

print("Missing values per module:")
print(df[score_columns].isnull().sum())

Missing values per module:
module_1_score     0
module_2_score     1
module_3_score     1
module_4_score     0
module_5_score     2
module_6_score     3
module_7_score     2
module_8_score     3
module_9_score     3
module_10_score    3
dtype: int64


In [6]:
# Convert module scores into a NumPy array
scores = df[score_columns].to_numpy(dtype=float)

# Count available module scores for each learner
available_scores = np.sum(~np.isnan(scores), axis=1)

# Calculate the average of available scores
skill_mastery_index = (
    np.nansum(scores, axis=1) /
    np.where(available_scores > 0, available_scores, np.nan)
)

# Add the new feature to the dataset
df["Skill_Mastery_Index"] = skill_mastery_index

display(
    df[["learner_id", "Skill_Mastery_Index"]].head(10)
)

,learner_id,Skill_Mastery_Index
0,LRN-1001,90.200000
1,LRN-1002,73.555556
2,LRN-1003,94.700000
3,LRN-1004,58.900000
4,LRN-1005,83.000000
5,LRN-1006,88.900000
6,LRN-1007,41.250000
7,LRN-1008,90.700000
8,LRN-1009,97.100000
9,LRN-1010,78.000000


In [7]:
# Classify learners based on their Skill Mastery Index
df["Performance_Category"] = np.select(
    [
        df["Skill_Mastery_Index"] >= 85,
        df["Skill_Mastery_Index"] >= 70
    ],
    [
        "High",
        "Medium"
    ],
    default="Low"
)

# Difference between module average and capstone score
df["Performance_Gap"] = (
    df["Skill_Mastery_Index"] - df["capstone_score"]
)

display(
    df[
        [
            "learner_id",
            "Skill_Mastery_Index",
            "capstone_score",
            "Performance_Category",
            "Performance_Gap"
        ]
    ].head(10)
)

,learner_id,Skill_Mastery_Index,capstone_score,Performance_Category,Performance_Gap
0,LRN-1001,90.200000,93.0,High,-2.800000
1,LRN-1002,73.555556,76.0,Medium,-2.444444
2,LRN-1003,94.700000,96.0,High,-1.300000
3,LRN-1004,58.900000,58.0,Low,0.900000
4,LRN-1005,83.000000,85.0,Medium,-2.000000
5,LRN-1006,88.900000,90.0,High,-1.100000
6,LRN-1007,41.250000,NaN,Low,NaN
7,LRN-1008,90.700000,91.0,High,-0.300000
8,LRN-1009,97.100000,99.0,High,-1.900000
9,LRN-1010,78.000000,79.0,Medium,-1.000000


In [8]:
# Average score for each module, ignoring missing values
module_averages = np.nanmean(scores, axis=0)

print("Average score for each module:")
for i, average in enumerate(module_averages, start=1):
    print(f"Module {i}: {average:.2f}")

# Correlation matrix between module scores
correlation_matrix = np.corrcoef(
    np.where(np.isnan(scores), np.nanmean(scores, axis=0), scores),
    rowvar=False
)

print("\nCorrelation matrix shape:", correlation_matrix.shape)

Average score for each module:
Module 1: 79.58
Module 2: 79.09
Module 3: 80.30
Module 4: 80.12
Module 5: 85.18
Module 6: 85.29
Module 7: 82.68
Module 8: 84.95
Module 9: 86.52
Module 10: 86.24

Correlation matrix shape: (10, 10)


In [9]:
# Check feature values and missing data
print("Dataset shape:", df.shape)

print("\nMissing Skill_Mastery_Index values:",
      df["Skill_Mastery_Index"].isna().sum())

print("\nPerformance categories:")
print(df["Performance_Category"].value_counts())

print("\nSkill Mastery Index summary:")
print(df["Skill_Mastery_Index"].describe().round(2))

# Verify the index stays within the expected range
valid_index = df["Skill_Mastery_Index"].dropna()

print("\nAll mastery scores between 0 and 100:",
      valid_index.between(0, 100).all())

Dataset shape: (24, 21)

Missing Skill_Mastery_Index values: 0

Performance categories:
Performance_Category
High      13
Medium     7
Low        4
Name: count, dtype: int64

Skill Mastery Index summary:
count    24.00
mean     80.60
std      15.30
min      41.25
25%      73.44
50%      85.80
75%      90.82
max      97.10
Name: Skill_Mastery_Index, dtype: float64

All mastery scores between 0 and 100: True


In [10]:
df.to_csv("sprints_cohort_engineered.csv", index=False)

print("Engineered dataset saved successfully!")

Engineered dataset saved successfully!


In [11]:
import json
import pandas as pd
import numpy as np

file_path = "/kaggle/input/datasets/janadarwish/sprints-dataset/employer_partner_requirements.json"

with open(file_path, "r", encoding="utf-8") as file:
    requirements_data = json.load(file)

requirements_df = pd.DataFrame(
    requirements_data["employer_requirements"]
)

print("Total employer roles:", len(requirements_df))
print("Available columns:", requirements_df.columns.tolist())

display(
    requirements_df[
        ["role_id", "job_title", "required_skills",
         "minimum_assessment_threshold", "priority_weight"]
    ].head()
)

Total employer roles: 50
Available columns: ['role_id', 'job_title', 'company_tier', 'industry', 'required_skills', 'minimum_assessment_threshold', 'priority_weight']


,role_id,job_title,required_skills,minimum_assessment_threshold,priority_weight
0,REQ-1001,Junior Data Analyst,"[Python, Pandas, SQL, Data Visualization]",85,0.90
1,REQ-1002,Data Scientist I,"[Python, Machine Learning, Scikit-Learn, NumPy...",92,0.95
2,REQ-1003,BI Developer,"[SQL, Tableau, Data Visualization, Python]",78,0.70
3,REQ-1004,Product Analyst,"[Python, A/B Testing, Pandas, Plotly]",88,0.85
4,REQ-1005,Analytics Engineer,"[Python, ETL, SQL, Pandas]",80,0.75


In [12]:
import json

requirements_path = (
    "/kaggle/input/datasets/janadarwish/"
    "sprints-dataset/employer_partner_requirements.json"
)

with open(requirements_path, "r", encoding="utf-8") as file:
    requirements = json.load(file)["employer_requirements"]

jobs_df = pd.DataFrame(requirements)

# Use the existing Skill_Mastery_Index from Notebook 02
learner_scores = df["Skill_Mastery_Index"].to_numpy(dtype=float)

thresholds = jobs_df["minimum_assessment_threshold"].to_numpy(dtype=float)
priorities = jobs_df["priority_weight"].to_numpy(dtype=float)

# Compare every learner with every job threshold
eligible = learner_scores[:, None] >= thresholds[None, :]

# Assessment score relative to each job's minimum threshold.
# This is NOT a direct technical-skill similarity score.
assessment_match = np.clip(
    learner_scores[:, None] / thresholds[None, :] * 100,
    0,
    100
)

# Apply the employer priority weights
weighted_match = assessment_match * priorities[None, :]

# Keep the results in a long-format DataFrame
match_df = pd.DataFrame({
    "learner_id": np.repeat(df["learner_id"].to_numpy(), len(jobs_df)),
    "role_id": np.tile(jobs_df["role_id"].to_numpy(), len(df)),
    "job_title": np.tile(jobs_df["job_title"].to_numpy(), len(df)),
    "assessment_score": np.repeat(learner_scores, len(jobs_df)),
    "minimum_threshold": np.tile(thresholds, len(df)),
    "priority_weight": np.tile(priorities, len(df)),
    "meets_threshold": eligible.ravel(),
    "assessment_match_score": assessment_match.ravel(),
    "priority_weighted_score": weighted_match.ravel()
})

print("Learner-job comparisons:", len(match_df))
print("Learners:", df["learner_id"].nunique())
print("Employer roles:", len(jobs_df))

display(
    match_df.sort_values(
        ["learner_id", "priority_weighted_score"],
        ascending=[True, False]
    ).groupby("learner_id").head(3)
)

Learner-job comparisons: 1200
Learners: 24
Employer roles: 50


,learner_id,role_id,job_title,assessment_score,minimum_threshold,priority_weight,meets_threshold,assessment_match_score,priority_weighted_score
8,LRN-1001,REQ-1009,Data Engineer I,90.200000,90.0,0.95,True,100.000000,95.000000
6,LRN-1001,REQ-1007,Machine Learning Engineer,90.200000,95.0,1.00,False,94.947368,94.947368
49,LRN-1001,REQ-1050,Chief Data Officer Apprentice,90.200000,95.0,1.00,False,94.947368,94.947368
50,LRN-1002,REQ-1001,Junior Data Analyst,73.555556,85.0,0.90,False,86.535948,77.882353
58,LRN-1002,REQ-1009,Data Engineer I,73.555556,90.0,0.95,False,81.728395,77.641975
...,...,...,...,...,...,...,...,...,...
1108,LRN-1023,REQ-1009,Data Engineer I,45.750000,90.0,0.95,False,50.833333,48.291667
1106,LRN-1023,REQ-1007,Machine Learning Engineer,45.750000,95.0,1.00,False,48.157895,48.157895
1156,LRN-1024,REQ-1007,Machine Learning Engineer,91.200000,95.0,1.00,False,96.000000,96.000000
1199,LRN-1024,REQ-1050,Chief Data Officer Apprentice,91.200000,95.0,1.00,False,96.000000,96.000000


In [13]:
# Keep only learners who meet the minimum assessment threshold
eligible_matches = match_df[match_df["meets_threshold"]].copy()

# Rank eligible jobs for each learner by employer priority
eligible_matches = eligible_matches.sort_values(
    ["learner_id", "priority_weight", "assessment_score"],
    ascending=[True, False, False]
)

print("Eligible learner-job matches:", len(eligible_matches))

display(
    eligible_matches.groupby("learner_id").head(3)
)

Eligible learner-job matches: 612


,learner_id,role_id,job_title,assessment_score,minimum_threshold,priority_weight,meets_threshold,assessment_match_score,priority_weighted_score
8,LRN-1001,REQ-1009,Data Engineer I,90.200000,90.0,0.95,True,100.0,95.0
35,LRN-1001,REQ-1036,Cybersecurity Data Analyst,90.200000,90.0,0.92,True,100.0,92.0
0,LRN-1001,REQ-1001,Junior Data Analyst,90.200000,85.0,0.90,True,100.0,90.0
60,LRN-1002,REQ-1011,Customer Insights Analyst,73.555556,72.0,0.50,True,100.0,50.0
86,LRN-1002,REQ-1037,Market Research Analyst,73.555556,71.0,0.50,True,100.0,50.0
65,LRN-1002,REQ-1016,Junior Python Developer (Data),73.555556,70.0,0.45,True,100.0,45.0
109,LRN-1003,REQ-1010,Junior Quantitative Analyst,94.700000,94.0,0.98,True,100.0,98.0
144,LRN-1003,REQ-1045,Recommendation Systems Engineer,94.700000,94.0,0.96,True,100.0,96.0
101,LRN-1003,REQ-1002,Data Scientist I,94.700000,92.0,0.95,True,100.0,95.0
207,LRN-1005,REQ-1008,Healthcare Data Analyst,83.000000,82.0,0.80,True,100.0,80.0
